#### Bibliotecas

In [1]:
import numpy as np
import pandas as pd

In [7]:
import sys
!{sys.executable} -m pip install xlrd

  Using cached xlrd-2.0.2-py2.py3-none-any.whl.metadata (3.5 kB)
Using cached xlrd-2.0.2-py2.py3-none-any.whl (96 kB)


#### Base de dados

In [9]:
base = r"C:\Users\agcsd\Downloads\default of credit card clients.xls"
df_base = pd.read_excel(base)
df_base.head()

,Unnamed: 0,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X15,X16,X17,X18,X19,X20,X21,X22,X23,Y
0,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
1,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
2,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
3,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
4,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0


#### Substituição de header pela primeira linha

In [11]:
# Substituir header pela primeira linha
df_base.columns = df_base.iloc[0]   # pega a linha 0 como header
df_base = df_base[1:]               # remove a linha 0 do dataset, que agora virou header

# Resetar índice (opcional)
df_base = df_base.reset_index(drop=True)

# Conferir
df_base.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [22]:
# Conferir
df_base.count()

0
ID                            30000
LIMIT_BAL                     30000
SEX                           30000
EDUCATION                     30000
MARRIAGE                      30000
AGE                           30000
PAY_0                         30000
PAY_2                         30000
PAY_3                         30000
PAY_4                         30000
PAY_5                         30000
PAY_6                         30000
BILL_AMT1                     30000
BILL_AMT2                     30000
BILL_AMT3                     30000
BILL_AMT4                     30000
BILL_AMT5                     30000
BILL_AMT6                     30000
PAY_AMT1                      30000
PAY_AMT2                      30000
PAY_AMT3                      30000
PAY_AMT4                      30000
PAY_AMT5                      30000
PAY_AMT6                      30000
default payment next month    30000
dtype: int64

#### Cópia do df_base

In [28]:
# 🔹 0. criar df_features a partir de df_base.copy()
df_features = df_base.copy()

#### Variáveis Demográficas

##### Quadrática

In [30]:
# 🔹 1. limpar possíveis duplicações antigas
df_features = df_features.loc[:, ~df_features.columns.str.endswith(('_x', '_y'))]

# 🔹 2. garantir que AGE existe no df_features
if 'AGE' not in df_features.columns:
    df_features = df_features.merge(
        df_base[['ID', 'AGE']],
        on='ID',
        how='left'
    )

# 🔹 3. criar AGE_SQ diretamente no df_features
df_features['AGE_SQ'] = df_features['AGE'].astype(float) ** 2

# 🔹 4. reposicionar AGE_SQ antes de "default payment next month"
colunas = list(df_features.columns)
if "default payment next month" in colunas:
    idx = colunas.index("default payment next month")
    # remove AGE_SQ e insere na posição correta
    colunas.remove("AGE_SQ")
    colunas.insert(idx, "AGE_SQ")
    df_features = df_features[colunas]

# 🔹 5. validação
df_features.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,AGE_SQ,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,689,0,0,0,0,576.0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3455,3261,0,1000,1000,1000,0,2000,676.0,1
2,3,90000,2,2,2,34,0,0,0,0,...,14948,15549,1518,1500,1000,1000,1000,5000,1156.0,0
3,4,50000,2,2,1,37,0,0,0,0,...,28959,29547,2000,2019,1200,1100,1069,1000,1369.0,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,19146,19131,2000,36681,10000,9000,689,679,3249.0,0


#### Faixas de Idade

##### Mínimo: 21
Máximo: 79

Média: ~35,5

Quartis:

Q1 = 28

Q2 (mediana) = 34

Q3 = 41

Faixa 1: Jovens → até 28 anos (AGE < 28) Faixa 2: Adultos jovens → 28 a 34 anos Faixa 3: Adultos maduros → 34 a 41 anos Faixa 4: Idosos → acima de 41 anos

In [33]:
df_features['AGE'] = df_features['AGE'].astype(int)  # garantir numérico
print(df_features['AGE'].describe())             # resumo estatístico
print(df_features['AGE'].quantile([0.25, 0.5, 0.75]))

count    30000.000000
mean        35.485500
std          9.217904
min         21.000000
25%         28.000000
50%         34.000000
75%         41.000000
max         79.000000
Name: AGE, dtype: float64
0.25    28.0
0.50    34.0
0.75    41.0
Name: AGE, dtype: float64


In [38]:
df_features['AGE_LESS_28']   = (df_features['AGE'] < 28).astype(int)
df_features['AGE_28_34']     = ((df_features['AGE'] >= 28) & (df_base['AGE'] < 34)).astype(int)
df_features['AGE_34_41']     = ((df_features['AGE'] >= 34) & (df_base['AGE'] < 41)).astype(int)
df_features['AGE_41_PLUS']   = (df_features['AGE'] >= 41).astype(int)

df_features[['AGE','AGE_LESS_28','AGE_28_34','AGE_34_41','AGE_41_PLUS']].head(10)

,AGE,AGE_LESS_28,AGE_28_34,AGE_34_41,AGE_41_PLUS
0,24,1,0,0,0
1,26,1,0,0,0
2,34,0,0,1,0
3,37,0,0,1,0
4,57,0,0,0,1
5,37,0,0,1,0
6,29,0,1,0,0
7,23,1,0,0,0
8,28,0,1,0,0
9,35,0,0,1,0


#### Transformações não lineares

##### Histórico de atrasos (PAY_0–PAY_6)

##### Variáveis com Janelas Temporais Ajustadas

In [55]:
pay_cols = ['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']

# mapear blocos de 2 meses
blocos = {
    'B1': pay_cols[0:2],  # M1-M2
    'B2': pay_cols[2:4],  # M3-M4
    'B3': pay_cols[4:6]   # M5-M6
}

for bloco, cols in blocos.items():

    # 🔹 contagem de atrasos (PAY > 0)
    df_features[f'ATRASOS_CONTAGEM_{bloco}'] = (df_features[cols] > 0).sum(axis=1)

    # 🔹 média dos PAY
    df_features[f'ATRASO_MEDIO_{bloco}'] = df_features[cols].mean(axis=1)

    # 🔹 volatilidade
    df_features[f'ATRASO_VOLATIL_{bloco}'] = df_features[cols].std(axis=1)

df_features

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,ATRASO_VOLATIL_B1,ATRASOS_FAIXA_B1,ATRASOS_CONTAGEM_B2,ATRASO_MEDIO_B2,ATRASO_VOLATIL_B2,ATRASOS_FAIXA_B2,ATRASOS_CONTAGEM_B3,ATRASO_MEDIO_B3,ATRASO_VOLATIL_B3,ATRASOS_FAIXA_B3
0,1,20000,2,2,1,24,2,2,-1,-1,...,0.0,2,0,-1.0,0.0,0,0,-2.0,0.0,0
1,2,120000,2,2,2,26,-1,2,0,0,...,2.12132,1,0,0.0,0.0,0,1,1.0,1.414214,1
2,3,90000,2,2,2,34,0,0,0,0,...,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0
3,4,50000,2,2,1,37,0,0,0,0,...,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,0.707107,0,0,-0.5,0.707107,0,0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,220000,1,3,1,39,0,0,0,0,...,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0
29996,29997,150000,1,3,2,43,-1,-1,-1,-1,...,0.0,0,0,-1.0,0.0,0,0,0.0,0.0,0
29997,29998,30000,1,2,2,37,4,3,2,-1,...,0.707107,2,1,0.5,2.12132,1,0,0.0,0.0,0
29998,29999,80000,1,3,1,41,1,-1,0,0,...,1.414214,1,0,0.0,0.0,0,0,-0.5,0.707107,0


##### Limite de Crédito

#### 
UTIL_B1, UTIL_B2, UTIL_B3 (explicação pronta)
##### UTIL_B1
UTIL_B1=(BILL_AMT1+BILL_AMT2)/2LIMIT_BALUTIL\_B1 = \frac{(BILL\_AMT1 + BILL\_AMT2)/2}{LIMIT\_BAL}UTIL_B1=LIMIT_BAL(BILL_AMT1+BILL_AMT2)/2 
= mede a utilização média do limite de crédito no período inicial (M1–M2), representando o comportamento base do cliente no começo da janela observada.
________________________________________
##### UTIL_B2
UTIL_B2=(BILL_AMT3+BILL_AMT4)/2LIMIT_BALUTIL\_B2 = \frac{(BILL\_AMT3 + BILL\_AMT4)/2}{LIMIT\_BAL}UTIL_B2=LIMIT_BAL(BILL_AMT3+BILL_AMT4)/2 
= mede a utilização média do limite no período intermediário (M3–M4), capturando possível mudança de comportamento em relação ao início.
________________________________________
##### UTIL_B3
UTIL_B3=(BILL_AMT5+BILL_AMT6)/2LIMIT_BALUTIL\_B3 = \frac{(BILL\_AMT5 + BILL\_AMT6)/2}{LIMIT\_BAL}UTIL_B3=LIMIT_BAL(BILL_AMT5+BILL_AMT6)/2 
= mede a utilização média do limite no período mais recente (M5–M6), sendo o indicador mais próximo do comportamento atual do cliente antes da previsão de risco.
________________________________________
🧠 Interpretação geral (igual ao estilo que você já usou)
•	Valores mais altos → maior uso do crédito em cada período 
•	Valores mais baixos → menor utilização do limite 
•	Comparação entre blocos → evolução do consumo de crédito ao longo do tempo 
________________________________________
🔥 Leitura correta do conjunto
•	UTIL_B1 → baseline comportamental 
•	UTIL_B2 → adaptação ou estabilidade intermediária 
•	UTIL_B3 → estado atual do risco (mais relevante para default em M7)
ial.

In [66]:
import numpy as np

# garantir numérico
df_features['LIMIT_BAL'] = df_features['LIMIT_BAL'].astype(float)

# =========================
# 🔹 UTILIZAÇÃO POR BLOCO
# =========================

df_features['UTIL_B1'] = df_features[['BILL_AMT1', 'BILL_AMT2']].mean(axis=1) / df_features['LIMIT_BAL']
df_features['UTIL_B2'] = df_features[['BILL_AMT3', 'BILL_AMT4']].mean(axis=1) / df_features['LIMIT_BAL']
df_features['UTIL_B3'] = df_features[['BILL_AMT5', 'BILL_AMT6']].mean(axis=1) / df_features['LIMIT_BAL']

df_features[['UTIL_B1','UTIL_B2','UTIL_B3']]

,UTIL_B1,UTIL_B2,UTIL_B3
0,0.175375,0.017225,0.0
1,0.018363,0.024808,0.027983
2,0.240367,0.154944,0.169428
3,0.95223,0.77605,0.58506
4,0.14287,0.56775,0.38277
...,...,...,...
29995,0.867643,0.673566,0.107311
29996,0.011703,0.041603,0.0173
29997,0.11535,0.393933,0.66565
29998,0.479587,0.806737,0.379994


#### Tendências | Regressão Linear Univariada

In [86]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Calcular a taxa de utilização mês a mês
for i in range(1,7):
    df_features[f'UTILIZACAO_{i}'] = df_features[f'BILL_AMT{i}'] / df_features['LIMIT_BAL']
    
util_cols_all = [
    'UTILIZACAO_1','UTILIZACAO_2','UTILIZACAO_3',
    'UTILIZACAO_4','UTILIZACAO_5','UTILIZACAO_6'
]

df_features[util_cols_all] = df_features[util_cols_all].astype(float)

# 🔹 DEFINIR BLOCOS (2M)
util_blocos = {
    'B1': ['UTILIZACAO_1', 'UTILIZACAO_2'],
    'B2': ['UTILIZACAO_3', 'UTILIZACAO_4'],
    'B3': ['UTILIZACAO_5', 'UTILIZACAO_6']
}

# eixo temporal dentro do bloco
X = np.array([[1], [2]])

# -----------------------------
# 🔹 FUNÇÃO SEGURA
# -----------------------------
def calc_tendencia_bloco(row, cols):
    y = row[cols].astype(float).values

    # segurança contra NaN
    if np.isnan(y).any():
        y = np.nan_to_num(y, nan=0.0)

    # caso degenerado (todos iguais)
    if np.all(y == y[0]):
        return 0.0

    model = LinearRegression().fit(X, y)
    return model.coef_[0]

# -----------------------------
# 🔹 APLICAR POR BLOCO
# -----------------------------
for bloco, cols in util_blocos.items():
    df_features[f'TENDENCIA_UTIL_{bloco}'] = df_features.apply(
        lambda row: calc_tendencia_bloco(row, cols),
        axis=1
    )

df_features[
    ['TENDENCIA_UTIL_B1',
     'TENDENCIA_UTIL_B2',
     'TENDENCIA_UTIL_B3']]

,TENDENCIA_UTIL_B1,TENDENCIA_UTIL_B2,TENDENCIA_UTIL_B3
0,-0.040550,-0.034450,0.000000
1,-0.007975,0.004917,-0.001617
2,-0.169022,0.008578,0.006678
3,0.024860,-0.419540,0.011760
4,-0.058940,-0.297900,-0.000300
...,...,...,...
29995,0.017577,-0.547095,-0.069350
29996,0.000967,0.036513,-0.034600
29997,-0.006967,0.604000,-0.040833
29998,1.000300,-0.294125,0.463612


#### Volatilidade | Tendências  | Regressão Linear Univariada

In [93]:
import numpy as np

'''# -----------------------------
# 🔹 UTILIZAÇÃO MENSAL
# -----------------------------
for i in range(1, 7):
    df_features[f'UTILIZACAO_{i}'] = (
        df_features[f'BILL_AMT{i}'] / df_features['LIMIT_BAL']
    ).astype(float)

# garantir tipo numérico
util_cols_all = [f'UTILIZACAO_{i}' for i in range(1, 7)]
df_features[util_cols_all] = df_features[util_cols_all].astype(float)
'''

# -----------------------------
# 🔹 BLOCOS 2M
# -----------------------------
util_blocos = {
    'B1': ['UTILIZACAO_1', 'UTILIZACAO_2'],
    'B2': ['UTILIZACAO_3', 'UTILIZACAO_4'],
    'B3': ['UTILIZACAO_5', 'UTILIZACAO_6']
}

# -----------------------------
# 🔹 VOLATILIDADE POR BLOCO
# -----------------------------
for bloco, cols in util_blocos.items():
    df_features[f'UTIL_VOLATIL_{bloco}'] = df_features[cols].std(axis=1)

# -----------------------------
# 🔹 VISUALIZAÇÃO
# -----------------------------
cols_show = (
    util_cols_all +
    [f'UTIL_VOLATIL_{b}' for b in util_blocos.keys()]
)


df_features[
    ['UTIL_VOLATIL_B1',
     'UTIL_VOLATIL_B2',
     'UTIL_VOLATIL_B3']]

,UTIL_VOLATIL_B1,UTIL_VOLATIL_B2,UTIL_VOLATIL_B3
0,0.028673,0.024360,0.000000
1,0.005639,0.003477,0.001143
2,0.119517,0.006065,0.004722
3,0.017579,0.296660,0.008316
4,0.041677,0.210647,0.000212
...,...,...,...
29995,0.012429,0.386855,0.049038
29996,0.000684,0.025819,0.024466
29997,0.004926,0.427092,0.028874
29998,0.707319,0.207978,0.327824


In [95]:
# -----------------------------
# 🔹 FLAG DE ZERO POR BLOCO
# -----------------------------
for bloco, cols in util_blocos.items():
    df_features[f'UTIL_ZERO_{bloco}'] = (
        df_features[cols].mean(axis=1) == 0
    ).astype(int)

# -----------------------------
# 🔹 CONCENTRAÇÃO POR BLOCO
# -----------------------------
for bloco, cols in util_blocos.items():
    media = df_features[cols].mean(axis=1)
    std = df_features[cols].std(axis=1)
    
    df_features[f'CONCENTRACAO_UTIL_{bloco}'] = (
        std / media.replace(0, np.nan)
    ).fillna(0)

# -----------------------------
# 🔹 VISUALIZAÇÃO
# -----------------------------
cols_show = []
for b in util_blocos.keys():
    cols_show += [f'UTIL_ZERO_{b}', f'CONCENTRACAO_UTIL_{b}']

df_features[
    ['UTIL_ZERO_B1',
     'UTIL_ZERO_B2',
     'UTIL_ZERO_B3',
     'CONCENTRACAO_UTIL_B1',
     'CONCENTRACAO_UTIL_B2',
     'CONCENTRACAO_UTIL_B3']]

,UTIL_ZERO_B1,UTIL_ZERO_B2,UTIL_ZERO_B3,CONCENTRACAO_UTIL_B1,CONCENTRACAO_UTIL_B2,CONCENTRACAO_UTIL_B3
0,0,0,1,0.163496,1.414214,0.000000
1,0,0,0,0.307103,0.140139,0.040851
2,0,0,0,0.497227,0.039146,0.027870
3,0,0,0,0.018461,0.382269,0.014213
4,0,0,0,0.291712,0.371021,0.000554
...,...,...,...,...,...,...
29995,0,0,0,0.014325,0.574339,0.456968
29996,0,0,0,0.058405,0.620595,1.414214
29997,0,0,0,0.042706,1.084175,0.043376
29998,0,0,0,1.474849,0.257801,0.862708


#### Proporção de Pagamento e Proporção de Pagamento Volátil

In [104]:
# -----------------------------
# 🔹 PROPORÇÃO DE PAGAMENTO (MENSAL)
# -----------------------------
prop_cols = []

for i in range(1, 7):
    col_name = f'PROP_PAGAMENTO_{i}'
    
    bill = df_features[f'BILL_AMT{i}'].astype(float)
    bill = bill.mask(bill == 0, np.nan)
    pay = df_features[f'PAY_AMT{i}']
    
    df_features[col_name] = (pay / bill).astype(float).fillna(0)
    prop_cols.append(col_name)

# -----------------------------
# 🔹 BLOCOS (2M)
# -----------------------------
prop_blocos = {
    'B1': prop_cols[0:2],  # M1-M2
    'B2': prop_cols[2:4],  # M3-M4
    'B3': prop_cols[4:6]   # M5-M6
}

# -----------------------------
# 🔹 VOLATILIDADE POR BLOCO
# -----------------------------
for bloco, cols in prop_blocos.items():
    
    # volatilidade
    df_features[f'PROP_PAGAMENTO_VOLATIL_{bloco}'] = df_features[cols].std(axis=1)
    
    # log da volatilidade
    df_features[f'PROP_PAGAMENTO_VOLATIL_LOG_{bloco}'] = np.log1p(
        df_features[f'PROP_PAGAMENTO_VOLATIL_{bloco}']
    )

# -----------------------------
# 🔹 VISUALIZAÇÃO
# -----------------------------
cols_show = (
    prop_cols +
    [f'PROP_PAGAMENTO_VOLATIL_{b}' for b in prop_blocos.keys()] +
    [f'PROP_PAGAMENTO_VOLATIL_LOG_{b}' for b in prop_blocos.keys()]
)

df_features[
    ['PROP_PAGAMENTO_VOLATIL_B1',
     'PROP_PAGAMENTO_VOLATIL_B2',
     'PROP_PAGAMENTO_VOLATIL_B3',
     'PROP_PAGAMENTO_VOLATIL_LOG_B1',
     'PROP_PAGAMENTO_VOLATIL_LOG_B2',
     'PROP_PAGAMENTO_VOLATIL_LOG_B3']]

,PROP_PAGAMENTO_VOLATIL_B1,PROP_PAGAMENTO_VOLATIL_B2,PROP_PAGAMENTO_VOLATIL_B3,PROP_PAGAMENTO_VOLATIL_LOG_B1,PROP_PAGAMENTO_VOLATIL_LOG_B2,PROP_PAGAMENTO_VOLATIL_LOG_B3
0,0.157059,0.000000,0.000000,0.145881,0.000000,0.000000
1,0.409917,0.047541,0.433675,0.343531,0.046445,0.360241
2,0.038905,0.002809,0.180076,0.038167,0.002805,0.165579
3,0.000497,0.010256,0.002171,0.000497,0.010204,0.002168
4,4.410375,0.106591,0.000350,1.688318,0.101284,0.000350
...,...,...,...,...,...,...
29995,0.041536,0.007504,0.068935,0.040696,0.007476,0.066663
29996,0.592118,1.806673,0.000000,0.465065,1.032000,0.000000
29997,0.000000,5.498198,0.044531,0.000000,1.871525,0.043568
29998,36.955054,0.014890,3.133043,3.636403,0.014780,1.419014


#### Proporção de Pagamento e Log e Flags Tratadas

In [124]:
prop_cols_trat = []

for i in range(1,7):
    col_name = f'PROP_PAGAMENTO_TRAT_{i}'
    
    bill = df_features[f'BILL_AMT{i}'].astype(float)
    bill = bill.mask(bill == 0, np.nan)
    
    pay = df_features[f'PAY_AMT{i}'].astype(float)
    
    prop = pay.div(bill).fillna(0)
    
    df_features[f'PROP_NEGATIVA_FLAG_{i}'] = (prop < 0).astype(int)
    df_features[f'PROP_SUPERIOR_FLAG_{i}'] = (prop > 1).astype(int)
    
    df_features[col_name] = np.clip(prop, 0, 1)
    
    prop_cols_trat.append(col_name)
    
# -----------------------------
# 🔹 BLOCOS (2M)
# -----------------------------
prop_blocos = {
    'B1': prop_cols_trat[0:2],  # M1-M2
    'B2': prop_cols_trat[2:4],  # M3-M4
    'B3': prop_cols_trat[4:6]   # M5-M6
}

# -----------------------------
# 🔹 MÉDIA POR BLOCO
# -----------------------------
for bloco, cols in prop_blocos.items():
    df_features[f'PROP_PAGAMENTO_TRAT_{bloco}'] = df_features[cols].mean(axis=1)

# -----------------------------
# 🔹 VOLATILIDADE POR BLOCO
# -----------------------------
for bloco, cols in prop_blocos.items():
    df_features[f'PROP_PAGAMENTO_VOLATIL_TRAT_{bloco}'] = df_features[cols].std(axis=1)

# -----------------------------
# 🔹 FLAGS POR BLOCO (ANY)
# -----------------------------
for bloco, idxs in zip(['B1', 'B2', 'B3'], [(1, 2), (3, 4), (5, 6)]):

    df_features[f'PROP_NEGATIVA_FLAG_{bloco}'] = df_features[
        [f'PROP_NEGATIVA_FLAG_{i}' for i in idxs]
    ].max(axis=1)

    df_features[f'PROP_SUPERIOR_FLAG_{bloco}'] = df_features[
        [f'PROP_SUPERIOR_FLAG_{i}' for i in idxs]
    ].max(axis=1)

# -----------------------------
# 🔹 VISUALIZAÇÃO (DINÂMICA)
# -----------------------------
cols_show = []

for b in ['B1', 'B2', 'B3']:
    cols_show += [
        f'PROP_PAGAMENTO_TRAT_{b}',
        f'PROP_PAGAMENTO_VOLATIL_TRAT_{b}',
        f'PROP_NEGATIVA_FLAG_{b}',
        f'PROP_SUPERIOR_FLAG_{b}'
    ]
    
df_features[
    [
        'PROP_PAGAMENTO_TRAT_B1',
        'PROP_PAGAMENTO_TRAT_B2',
        'PROP_PAGAMENTO_TRAT_B3',
        'PROP_PAGAMENTO_VOLATIL_TRAT_B1',
        'PROP_PAGAMENTO_VOLATIL_TRAT_B2',
        'PROP_PAGAMENTO_VOLATIL_TRAT_B3',
        'PROP_NEGATIVA_FLAG_B1',
        'PROP_NEGATIVA_FLAG_B2',
        'PROP_NEGATIVA_FLAG_B3',
        'PROP_SUPERIOR_FLAG_B1',
        'PROP_SUPERIOR_FLAG_B2',
        'PROP_SUPERIOR_FLAG_B3'
    ]
]

,PROP_PAGAMENTO_TRAT_B1,PROP_PAGAMENTO_TRAT_B2,PROP_PAGAMENTO_TRAT_B3,PROP_PAGAMENTO_VOLATIL_TRAT_B1,PROP_PAGAMENTO_VOLATIL_TRAT_B2,PROP_PAGAMENTO_VOLATIL_TRAT_B3,PROP_NEGATIVA_FLAG_B1,PROP_NEGATIVA_FLAG_B2,PROP_NEGATIVA_FLAG_B3,PROP_SUPERIOR_FLAG_B1,PROP_SUPERIOR_FLAG_B2,PROP_SUPERIOR_FLAG_B3
0,0.111057,0.000000,0.000000,0.157059,0.000000,0.000000,0,0,0,0,0,0
1,0.289855,0.339240,0.306654,0.409917,0.047541,0.433675,0,0,0,0,0,0
2,0.079427,0.071765,0.194231,0.038905,0.002809,0.180076,0,0,0,0,0,0
3,0.042211,0.031598,0.035379,0.000497,0.010256,0.002171,0,0,0,0,0,0
4,0.616050,0.354428,0.035739,0.542988,0.106591,0.000350,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
29995,0.074356,0.029317,0.111322,0.041536,0.007504,0.068935,0,0,0,0,0,0
29996,1.000000,0.507183,0.000000,0.000000,0.696948,0.000000,0,0,0,1,1,0
29997,0.000000,0.600584,0.128661,0.000000,0.564859,0.044531,0,0,0,0,1,0
29998,0.021747,0.025967,0.518429,0.030755,0.014890,0.681044,1,0,0,0,0,1


##### PROP_PAGAMENTO e PROP_PAGAMENTO por BLOCO e Média da proporção de pagamento por BLOCO

##### Proporção em relação ao limite (LIMIT_BAL)

In [136]:
import numpy as np

# -----------------------------
# 🔹 PROPORÇÃO DE PAGAMENTO (MENSAL)
# -----------------------------
pay_cols = ['PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6']

prop_cols = []

for i, col in enumerate(pay_cols, start=1):
    
    limit = df_features['LIMIT_BAL'].astype(float)
    limit = limit.mask(limit == 0, np.nan)
    
    pay = df_features[col].astype(float)
    
    df_features[f'PROP_PAGAMENTO_{i}'] = (pay / limit).fillna(0)
    
    prop_cols.append(f'PROP_PAGAMENTO_{i}')

# -----------------------------
# 🔹 BLOCOS (2M)
# -----------------------------
prop_blocos = {
    'B1': prop_cols[0:2],  # M1-M2
    'B2': prop_cols[2:4],  # M3-M4
    'B3': prop_cols[4:6]   # M5-M6
}

# -----------------------------
# 🔹 MÉDIA POR BLOCO
# -----------------------------
for bloco, cols in prop_blocos.items():
    df_features[f'MEDIA_PROP_PAGAMENTO_{bloco}'] = df_features[cols].mean(axis=1)

# -----------------------------
# 🔹 VISUALIZAÇÃO
# -----------------------------
cols_show = (
    prop_cols +
    [f'MEDIA_PROP_PAGAMENTO_{b}' for b in ['B1','B2','B3']]
)

df_features[
    [
        'PROP_PAGAMENTO_1',
        'PROP_PAGAMENTO_2',
        'PROP_PAGAMENTO_3',
        'PROP_PAGAMENTO_4',
        'PROP_PAGAMENTO_5',
        'PROP_PAGAMENTO_6',
        'MEDIA_PROP_PAGAMENTO_B1',
        'MEDIA_PROP_PAGAMENTO_B2',
        'MEDIA_PROP_PAGAMENTO_B3',
    ]
]

,PROP_PAGAMENTO_1,PROP_PAGAMENTO_2,PROP_PAGAMENTO_3,PROP_PAGAMENTO_4,PROP_PAGAMENTO_5,PROP_PAGAMENTO_6,MEDIA_PROP_PAGAMENTO_B1,MEDIA_PROP_PAGAMENTO_B2,MEDIA_PROP_PAGAMENTO_B3
0,0.000000,0.034450,0.000000,0.000000,0.000000,0.000000,0.017225,0.000000,0.000000
1,0.000000,0.008333,0.008333,0.008333,0.000000,0.016667,0.004167,0.008333,0.008333
2,0.016867,0.016667,0.011111,0.011111,0.011111,0.055556,0.016767,0.011111,0.033333
3,0.040000,0.040380,0.024000,0.022000,0.021380,0.020000,0.040190,0.023000,0.020690
4,0.040000,0.733620,0.200000,0.180000,0.013780,0.013580,0.386810,0.190000,0.013680
...,...,...,...,...,...,...,...,...,...
29995,0.038636,0.090909,0.022741,0.013850,0.022727,0.004545,0.064773,0.018295,0.013636
29996,0.012247,0.023507,0.059987,0.000860,0.000000,0.000000,0.017877,0.030423,0.000000
29997,0.000000,0.000000,0.733333,0.140000,0.066667,0.103333,0.000000,0.436667,0.085000
29998,1.073750,0.042612,0.014725,0.024075,0.662050,0.022550,0.558181,0.019400,0.342300


##### Volatilidade da proporção de pagamento por Bloco

In [144]:
# -----------------------------
# 🔹 COLUNAS BASE
# -----------------------------
prop_cols = [f'PROP_PAGAMENTO_{i}' for i in range(1,7)]

# -----------------------------
# 🔹 VOLATILIDADE POR BLOCO
# -----------------------------
for bloco, cols in prop_blocos.items():
    df_features[f'VOL_PROP_PAGAMENTO_{bloco}'] = df_features[cols].std(axis=1)

# -----------------------------
# 🔹 VISUALIZAÇÃO
# -----------------------------
cols_show = [f'VOL_PROP_PAGAMENTO_{b}' for b in ['B1','B2','B3']]

df_features[
    [
        'VOL_PROP_PAGAMENTO_B1',
        'VOL_PROP_PAGAMENTO_B2',
        'VOL_PROP_PAGAMENTO_B3',
    ]
]

,VOL_PROP_PAGAMENTO_B1,VOL_PROP_PAGAMENTO_B2,VOL_PROP_PAGAMENTO_B3
0,0.024360,0.000000,0.000000
1,0.005893,0.000000,0.011785
2,0.000141,0.000000,0.031427
3,0.000269,0.001414,0.000976
4,0.490463,0.014142,0.000141
...,...,...,...
29995,0.036962,0.006287,0.012856
29996,0.007962,0.041809,0.000000
29997,0.000000,0.419550,0.025927
29998,0.729124,0.006611,0.452195


##### MEDIA_PROP_PAGAMENTO_VOLATIL por BLOCO

In [146]:
# -----------------------------
# 🔹 COLUNAS BASE
# -----------------------------
prop_cols = [f'PROP_PAGAMENTO_{i}' for i in range(1,7)]

# -----------------------------
# 🔹 BLOCOS (2M)
# -----------------------------
prop_blocos = {
    'B1': prop_cols[0:2],  # M1-M2
    'B2': prop_cols[2:4],  # M3-M4
    'B3': prop_cols[4:6]   # M5-M6
}

# -----------------------------
# 🔹 MÉDIA POR BLOCO (nome correto)
# -----------------------------
for bloco, cols in prop_blocos.items():
    df_features[f'MEDIA_PROP_PAGAMENTO_{bloco}'] = df_features[cols].mean(axis=1)

# -----------------------------
# 🔹 VISUALIZAÇÃO
# -----------------------------
df_features[
    [
        'MEDIA_PROP_PAGAMENTO_B1',
        'MEDIA_PROP_PAGAMENTO_B2',
        'MEDIA_PROP_PAGAMENTO_B3',
    ]
]

,MEDIA_PROP_PAGAMENTO_B1,MEDIA_PROP_PAGAMENTO_B2,MEDIA_PROP_PAGAMENTO_B3
0,0.017225,0.000000,0.000000
1,0.004167,0.008333,0.008333
2,0.016767,0.011111,0.033333
3,0.040190,0.023000,0.020690
4,0.386810,0.190000,0.013680
...,...,...,...
29995,0.064773,0.018295,0.013636
29996,0.017877,0.030423,0.000000
29997,0.000000,0.436667,0.085000
29998,0.558181,0.019400,0.342300


##### Tendência de Endividamento

##### Perfis de Risco

##### perfil_risco_endividamento_B1 B2 e B3

In [172]:
# -----------------------------
# 🔹 PERFIL ENDIV + VOLATIL POR BLOCO
# -----------------------------

def classificar_perfil_endiv_volatil(tendencia, volatil, limiar_volatil=0.2):

    if tendencia > 0 and volatil < limiar_volatil:
        return "Esforço Consistente"       # dívida cresce, pagamentos regulares

    elif tendencia > 0 and volatil >= limiar_volatil:
        return "Instabilidade Crítica"     # dívida cresce e pagamentos irregulares

    elif tendencia < 0 and volatil < limiar_volatil:
        return "Recuperação Sustentada"    # dívida cai e pagamentos regulares

    elif tendencia < 0 and volatil >= limiar_volatil:
        return "Recuperação Instável"      # dívida cai mas pagamentos voláteis

    else:
        return "Estável"


# -----------------------------
# 🔹 BLOCOS
# -----------------------------
blocos = ['B1', 'B2', 'B3']

for b in blocos:

    tend_col = f'TEND_ENDIVIDAMENTO_{b}'

    # tenta usar volatilidade de pagamento se existir
    vol_col = f'PROP_PAGAMENTO_VOLATIL_{b}'

    # fallback seguro (caso não exista)
    if vol_col not in df_features.columns:
        vol_col = f'MEDIA_ENDIVIDAMENTO_{b}'  # fallback mais estável

    df_features[f'perfil_endiv_volatil_{b}'] = df_features.apply(
        lambda row: classificar_perfil_endiv_volatil(
            row[tend_col],
            row[vol_col]
        ),
        axis=1
    )

# -----------------------------
# 🔹 VISUALIZAÇÃO
# -----------------------------
df_features[
    [f'perfil_endiv_volatil_{b}' for b in blocos]
].head(10)

,perfil_endiv_volatil_B1,perfil_endiv_volatil_B2,perfil_endiv_volatil_B3
0,Recuperação Sustentada,Recuperação Sustentada,Estável
1,Recuperação Instável,Esforço Consistente,Recuperação Instável
2,Recuperação Sustentada,Esforço Consistente,Esforço Consistente
3,Esforço Consistente,Recuperação Sustentada,Esforço Consistente
4,Recuperação Instável,Recuperação Sustentada,Recuperação Sustentada
5,Recuperação Sustentada,Recuperação Sustentada,Esforço Consistente
6,Esforço Consistente,Esforço Consistente,Recuperação Sustentada
7,Recuperação Instável,Recuperação Instável,Instabilidade Crítica
8,Instabilidade Crítica,Esforço Consistente,Recuperação Sustentada
9,Estável,Estável,Esforço Consistente


##### % de ENDIVIDAMENTO, MÉDIA DE ENDIVIDAMENTO e TEND_ENDIVIDAMENTO

In [168]:
# -----------------------------
# 🔹 EVITA DUPLICAÇÃO DE COLUNAS (IMPORTANTE)
# -----------------------------
df_features = df_features.loc[:, ~df_features.columns.duplicated()].copy()

# -----------------------------
# 🔹 ENDIVIDAMENTO BASE
# -----------------------------
for i in range(1, 7):
    df_features[f'ENDIVIDAMENTO_{i}'] = (
        df_features[f'BILL_AMT{i}'].astype(float) /
        df_features['LIMIT_BAL'].astype(float).replace(0, np.nan)
    ).fillna(0)

endiv_cols = [f'ENDIVIDAMENTO_{i}' for i in range(1, 7)]
df_features[endiv_cols] = df_features[endiv_cols].astype(float)

# -----------------------------
# 🔹 BLOCOS (2M)
# -----------------------------
endiv_blocos = {
    'B1': endiv_cols[0:2],
    'B2': endiv_cols[2:4],
    'B3': endiv_cols[4:6]
}

# -----------------------------
# 🔹 DICIONÁRIOS (EVITA FRAGMENTAÇÃO)
# -----------------------------
media_dict = {}
trend_dict = {}

# -----------------------------
# 🔹 MÉDIA POR BLOCO
# -----------------------------
for bloco, cols in endiv_blocos.items():
    media_dict[f'MEDIA_ENDIVIDAMENTO_{bloco}'] = df_features[cols].mean(axis=1)

# -----------------------------
# 🔹 TENDÊNCIA POR BLOCO
# -----------------------------
x = np.array([1, 2])

def calc_tend_bloco(row, cols):
    y = row[cols].values.astype(float)

    if np.all(np.isnan(y)) or np.all(y == y[0]):
        return 0.0

    slope, _ = np.polyfit(x, y, 1)
    return slope

for bloco, cols in endiv_blocos.items():
    trend_dict[f'TEND_ENDIVIDAMENTO_{bloco}'] = df_features.apply(
        lambda row: calc_tend_bloco(row, cols),
        axis=1
    )

# -----------------------------
# 🔹 REMOVE SE JÁ EXISTIREM (SEGURANÇA EXTRA)
# -----------------------------
cols_to_drop = list(media_dict.keys()) + list(trend_dict.keys())
df_features = df_features.drop(columns=cols_to_drop, errors='ignore')

# -----------------------------
# 🔹 CONCAT FINAL (LIMPO)
# -----------------------------
df_features = pd.concat(
    [df_features,
     pd.DataFrame(media_dict),
     pd.DataFrame(trend_dict)],
    axis=1
)

# -----------------------------
# 🔹 GARANTE NOVAMENTE SEM DUPLICADOS
# -----------------------------
df_features = df_features.loc[:, ~df_features.columns.duplicated()].copy()

# -----------------------------
# 🔹 VISUALIZAÇÃO FINAL
# -----------------------------
df_features[
    endiv_cols +
    [f'MEDIA_ENDIVIDAMENTO_{b}' for b in ['B1','B2','B3']] +
    [f'TEND_ENDIVIDAMENTO_{b}' for b in ['B1','B2','B3']]
].head(10)

,ENDIVIDAMENTO_1,ENDIVIDAMENTO_2,ENDIVIDAMENTO_3,ENDIVIDAMENTO_4,ENDIVIDAMENTO_5,ENDIVIDAMENTO_6,MEDIA_ENDIVIDAMENTO_B1,MEDIA_ENDIVIDAMENTO_B2,MEDIA_ENDIVIDAMENTO_B3,TEND_ENDIVIDAMENTO_B1,TEND_ENDIVIDAMENTO_B2,TEND_ENDIVIDAMENTO_B3
0,0.195650,0.155100,0.034450,0.000000,0.000000,0.000000,0.175375,0.017225,0.000000,-0.040550,-0.034450,0.000000
1,0.022350,0.014375,0.022350,0.027267,0.028792,0.027175,0.018363,0.024808,0.027983,-0.007975,0.004917,-0.001617
2,0.324878,0.155856,0.150656,0.159233,0.166089,0.172767,0.240367,0.154944,0.169428,-0.169022,0.008578,0.006678
3,0.939800,0.964660,0.985820,0.566280,0.579180,0.590940,0.952230,0.776050,0.585060,0.024860,-0.419540,0.011760
4,0.172340,0.113400,0.716700,0.418800,0.382920,0.382620,0.142870,0.567750,0.382770,-0.058940,-0.297900,-0.000300
5,1.288000,1.141380,1.152160,0.387880,0.392380,0.400480,1.214690,0.770020,0.396430,-0.146620,-0.764280,0.008100
6,0.735930,0.824046,0.890014,1.085306,0.966006,0.947888,0.779988,0.987660,0.956947,0.088116,0.195292,-0.018118
7,0.118760,0.003800,0.006010,0.002210,-0.001590,0.005670,0.061280,0.004110,0.002040,-0.114960,-0.003800,0.007260
8,0.080607,0.100686,0.086486,0.087221,0.084236,0.026564,0.090646,0.086854,0.055400,0.020079,0.000736,-0.057671
9,0.000000,0.000000,0.000000,0.000000,0.650350,0.695600,0.000000,0.000000,0.672975,0.000000,0.000000,0.045250


In [176]:
# 🔹 transformar em DataFrame para melhor visualização
contagem = df_features.count().reset_index()
contagem.columns = ['coluna', 'n_non_null']
print(contagem)


                      coluna  n_non_null
0                         ID       30000
1                  LIMIT_BAL       30000
2                        SEX       30000
3                  EDUCATION       30000
4                   MARRIAGE       30000
..                       ...         ...
122    TEND_ENDIVIDAMENTO_B2       30000
123    TEND_ENDIVIDAMENTO_B3       30000
124  perfil_endiv_volatil_B1       30000
125  perfil_endiv_volatil_B2       30000
126  perfil_endiv_volatil_B3       30000

[127 rows x 2 columns]


In [178]:
import pandas as pd

# 🔹 garantir que todas as colunas sejam mostradas
pd.set_option('display.max_columns', None)

# 🔹 conferir contagem de não-nulos por coluna
print(df_features.count())


ID                         30000
LIMIT_BAL                  30000
SEX                        30000
EDUCATION                  30000
MARRIAGE                   30000
                           ...  
TEND_ENDIVIDAMENTO_B2      30000
TEND_ENDIVIDAMENTO_B3      30000
perfil_endiv_volatil_B1    30000
perfil_endiv_volatil_B2    30000
perfil_endiv_volatil_B3    30000
Length: 127, dtype: int64


In [208]:
# 🔹 mostrar todas as colunas
print(df_features.columns.tolist())


['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'AGE_SQ', 'default payment next month', 'AGE_LESS_28', 'AGE_28_34', 'AGE_34_41', 'AGE_41_PLUS', 'ATRASOS_CONTAGEM_B1', 'ATRASO_MEDIO_B1', 'ATRASO_VOLATIL_B1', 'ATRASOS_FAIXA_B1', 'ATRASOS_CONTAGEM_B2', 'ATRASO_MEDIO_B2', 'ATRASO_VOLATIL_B2', 'ATRASOS_FAIXA_B2', 'ATRASOS_CONTAGEM_B3', 'ATRASO_MEDIO_B3', 'ATRASO_VOLATIL_B3', 'ATRASOS_FAIXA_B3', 'UTIL_B1', 'UTIL_B2', 'UTIL_B3', 'UTILIZACAO_1', 'UTILIZACAO_2', 'UTILIZACAO_3', 'UTILIZACAO_4', 'UTILIZACAO_5', 'UTILIZACAO_6', 'TENDENCIA_UTILIZACAO', 'TENDENCIA_UTIL_B1', 'TENDENCIA_UTIL_B2', 'TENDENCIA_UTIL_B3', 'UTIL_VOLATIL_B1', 'UTIL_VOLATIL_B2', 'UTIL_VOLATIL_B3', 'UTIL_ZERO_B1', 'UTIL_ZERO_B2', 'UTIL_ZERO_B3', 'CONCENTRACAO_UTIL_B1', 'CONCENTRACAO_UTIL_B2', 'CONCENTRACAO_UTIL_B3',

In [180]:
# 🔹 mostrar todas as colunas
print(df_features.columns.tolist())


['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'AGE_SQ', 'default payment next month', 'AGE_LESS_28', 'AGE_28_34', 'AGE_34_41', 'AGE_41_PLUS', 'ATRASOS_CONTAGEM_B1', 'ATRASO_MEDIO_B1', 'ATRASO_VOLATIL_B1', 'ATRASOS_FAIXA_B1', 'ATRASOS_CONTAGEM_B2', 'ATRASO_MEDIO_B2', 'ATRASO_VOLATIL_B2', 'ATRASOS_FAIXA_B2', 'ATRASOS_CONTAGEM_B3', 'ATRASO_MEDIO_B3', 'ATRASO_VOLATIL_B3', 'ATRASOS_FAIXA_B3', 'UTIL_B1', 'UTIL_B2', 'UTIL_B3', 'UTILIZACAO_1', 'UTILIZACAO_2', 'UTILIZACAO_3', 'UTILIZACAO_4', 'UTILIZACAO_5', 'UTILIZACAO_6', 'TENDENCIA_UTILIZACAO', 'TENDENCIA_UTIL_B1', 'TENDENCIA_UTIL_B2', 'TENDENCIA_UTIL_B3', 'UTIL_VOLATIL_B1', 'UTIL_VOLATIL_B2', 'UTIL_VOLATIL_B3', 'UTIL_ZERO_B1', 'UTIL_ZERO_B2', 'UTIL_ZERO_B3', 'CONCENTRACAO_UTIL_B1', 'CONCENTRACAO_UTIL_B2', 'CONCENTRACAO_UTIL_B3',

In [228]:
# -----------------------------
# 🔹 REFERÊNCIAS
# -----------------------------
base_ref = df_base.copy()
feat_ref = df_features.copy()

# -----------------------------
# 🔹 BLOCOS (2M)
# -----------------------------
blocks = {
    "B1": (1, 2),
    "B2": (3, 4),
    "B3": (5, 6)
}

# -----------------------------
# 🔵 VARIÁVEIS GLOBAIS (BASE)
# -----------------------------
global_cols = [
    'ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
    'AGE_LESS_28', 'AGE_28_34', 'AGE_34_41', 'AGE_41_PLUS',
    'default payment next month'
]

# -----------------------------
# 🔹 FUNÇÃO SEGURA
# -----------------------------
def safe_cols(df, cols):
    return [c for c in cols if c in df.columns]

# -----------------------------
# 🔹 CONTAINER FINAL
# -----------------------------
dfs = {}

# -----------------------------
# 🔁 LOOP PRINCIPAL
# -----------------------------
for b, (start, end) in blocks.items():

    months = range(start, end + 1)

    # 🔵 df_base → dados brutos
    base_window_cols = (
        global_cols +
        [f'BILL_AMT{i}' for i in months] +
        [f'PAY_AMT{i}' for i in months]
    )

    # 🟢 df_features → features
    feature_window_cols = (
        ['AGE_SQ'] +  # 👈 agora vindo corretamente daqui
        [f'UTILIZACAO_{i}' for i in months] +
        [f'PROP_PAGAMENTO_{i}' for i in months] +
        [f'ENDIVIDAMENTO_{i}' for i in months] +
        [f'PROP_NEGATIVA_FLAG_{i}' for i in months] +
        [f'PROP_SUPERIOR_FLAG_{i}' for i in months] +
        [f'PROP_PAGAMENTO_TRAT_{i}' for i in months]
    )

    # 🔹 features agregadas por bloco
    derived_cols = [
        f'ATRASOS_CONTAGEM_{b}',
        f'ATRASO_MEDIO_{b}',
        f'ATRASO_VOLATIL_{b}',

        f'UTIL_B{b[-1]}',
        f'TENDENCIA_UTIL_B{b[-1]}',
        f'UTIL_VOLATIL_B{b[-1]}',
        f'UTIL_ZERO_B{b[-1]}',
        f'CONCENTRACAO_UTIL_B{b[-1]}',

        f'PROP_PAGAMENTO_VOLATIL_B{b[-1]}',
        f'PROP_PAGAMENTO_VOLATIL_LOG_B{b[-1]}',

        f'PROP_NEGATIVA_FLAG_B{b[-1]}',
        f'PROP_SUPERIOR_FLAG_B{b[-1]}',

        f'MEDIA_PROP_PAGAMENTO_B{b[-1]}',
        f'VOL_PROP_PAGAMENTO_B{b[-1]}',

        f'MEDIA_ENDIVIDAMENTO_B{b[-1]}',
        f'TEND_ENDIVIDAMENTO_B{b[-1]}'
    ]

    # -----------------------------
    # 🔹 SELEÇÃO SEGURA
    # -----------------------------
    base_cols_sel = safe_cols(base_ref, base_window_cols)
    feat_cols_sel = safe_cols(feat_ref, feature_window_cols + derived_cols)

    # -----------------------------
    # 🔹 MONTAGEM DO BLOCO
    # -----------------------------
    df_block = base_ref[base_cols_sel].copy()
    df_block[feat_cols_sel] = feat_ref[feat_cols_sel]

    # -----------------------------
    # 🔹 TARGET NO FINAL
    # -----------------------------
    target = 'default payment next month'
    cols = [c for c in df_block.columns if c != target] + [target]
    df_block = df_block[cols]

    # -----------------------------
    # 🔹 ARMAZENAR
    # -----------------------------
    dfs[f"df_features_{b.lower()}"] = df_block


# -----------------------------
# 🔹 OUTPUTS FINAIS
# -----------------------------
df_features_b1 = dfs["df_features_b1"]
df_features_b2 = dfs["df_features_b2"]
df_features_b3 = dfs["df_features_b3"]

In [230]:
df_features_b1

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,BILL_AMT1,BILL_AMT2,PAY_AMT1,PAY_AMT2,AGE_SQ,UTILIZACAO_1,UTILIZACAO_2,PROP_PAGAMENTO_1,PROP_PAGAMENTO_2,ENDIVIDAMENTO_1,ENDIVIDAMENTO_2,PROP_NEGATIVA_FLAG_1,PROP_NEGATIVA_FLAG_2,PROP_SUPERIOR_FLAG_1,PROP_SUPERIOR_FLAG_2,PROP_PAGAMENTO_TRAT_1,PROP_PAGAMENTO_TRAT_2,ATRASOS_CONTAGEM_B1,ATRASO_MEDIO_B1,ATRASO_VOLATIL_B1,UTIL_B1,TENDENCIA_UTIL_B1,UTIL_VOLATIL_B1,UTIL_ZERO_B1,CONCENTRACAO_UTIL_B1,PROP_PAGAMENTO_VOLATIL_B1,PROP_PAGAMENTO_VOLATIL_LOG_B1,PROP_NEGATIVA_FLAG_B1,PROP_SUPERIOR_FLAG_B1,MEDIA_PROP_PAGAMENTO_B1,VOL_PROP_PAGAMENTO_B1,MEDIA_ENDIVIDAMENTO_B1,TEND_ENDIVIDAMENTO_B1,default payment next month
0,1,20000,2,2,1,24,3913,3102,0,689,576.0,0.195650,0.155100,0.000000,0.034450,0.195650,0.155100,0,0,0,0,0.000000,0.222115,2,2.0,0.0,0.175375,-0.040550,0.028673,0,0.163496,0.157059,0.145881,0,0,0.017225,0.024360,0.175375,-0.040550,1
1,2,120000,2,2,2,26,2682,1725,0,1000,676.0,0.022350,0.014375,0.000000,0.008333,0.022350,0.014375,0,0,0,0,0.000000,0.579710,1,0.5,2.12132,0.018363,-0.007975,0.005639,0,0.307103,0.409917,0.343531,0,0,0.004167,0.005893,0.018363,-0.007975,1
2,3,90000,2,2,2,34,29239,14027,1518,1500,1156.0,0.324878,0.155856,0.016867,0.016667,0.324878,0.155856,0,0,0,0,0.051917,0.106937,0,0.0,0.0,0.240367,-0.169022,0.119517,0,0.497227,0.038905,0.038167,0,0,0.016767,0.000141,0.240367,-0.169022,0
3,4,50000,2,2,1,37,46990,48233,2000,2019,1369.0,0.939800,0.964660,0.040000,0.040380,0.939800,0.964660,0,0,0,0,0.042562,0.041859,0,0.0,0.0,0.95223,0.024860,0.017579,0,0.018461,0.000497,0.000497,0,0,0.040190,0.000269,0.952230,0.024860,0
4,5,50000,1,2,1,57,8617,5670,2000,36681,3249.0,0.172340,0.113400,0.040000,0.733620,0.172340,0.113400,0,0,0,1,0.232099,1.000000,0,-0.5,0.707107,0.14287,-0.058940,0.041677,0,0.291712,4.410375,1.688318,0,1,0.386810,0.490463,0.142870,-0.058940,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,220000,1,3,1,39,188948,192815,8500,20000,1521.0,0.858855,0.876432,0.038636,0.090909,0.858855,0.876432,0,0,0,0,0.044986,0.103726,0,0.0,0.0,0.867643,0.017577,0.012429,0,0.014325,0.041536,0.040696,0,0,0.064773,0.036962,0.867643,0.017577,0
29996,29997,150000,1,3,2,43,1683,1828,1837,3526,1849.0,0.011220,0.012187,0.012247,0.023507,0.011220,0.012187,0,0,1,1,1.000000,1.000000,0,-1.0,0.0,0.011703,0.000967,0.000684,0,0.058405,0.592118,0.465065,0,1,0.017877,0.007962,0.011703,0.000967,0
29997,29998,30000,1,2,2,37,3565,3356,0,0,1369.0,0.118833,0.111867,0.000000,0.000000,0.118833,0.111867,0,0,0,0,0.000000,0.000000,2,3.5,0.707107,0.11535,-0.006967,0.004926,0,0.042706,0.000000,0.000000,0,0,0.000000,0.000000,0.115350,-0.006967,1
29998,29999,80000,1,3,1,41,-1645,78379,85900,3409,1681.0,-0.020563,0.979738,1.073750,0.042612,-0.020563,0.979738,1,0,0,0,0.000000,0.043494,1,0.0,1.414214,0.479587,1.000300,0.707319,0,1.474849,36.955054,3.636403,1,0,0.558181,0.729124,0.479587,1.000300,1


In [234]:
df_features_b2

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,BILL_AMT3,BILL_AMT4,PAY_AMT3,PAY_AMT4,AGE_SQ,UTILIZACAO_3,UTILIZACAO_4,PROP_PAGAMENTO_3,PROP_PAGAMENTO_4,ENDIVIDAMENTO_3,ENDIVIDAMENTO_4,PROP_NEGATIVA_FLAG_3,PROP_NEGATIVA_FLAG_4,PROP_SUPERIOR_FLAG_3,PROP_SUPERIOR_FLAG_4,PROP_PAGAMENTO_TRAT_3,PROP_PAGAMENTO_TRAT_4,ATRASOS_CONTAGEM_B2,ATRASO_MEDIO_B2,ATRASO_VOLATIL_B2,UTIL_B2,TENDENCIA_UTIL_B2,UTIL_VOLATIL_B2,UTIL_ZERO_B2,CONCENTRACAO_UTIL_B2,PROP_PAGAMENTO_VOLATIL_B2,PROP_PAGAMENTO_VOLATIL_LOG_B2,PROP_NEGATIVA_FLAG_B2,PROP_SUPERIOR_FLAG_B2,MEDIA_PROP_PAGAMENTO_B2,VOL_PROP_PAGAMENTO_B2,MEDIA_ENDIVIDAMENTO_B2,TEND_ENDIVIDAMENTO_B2,default payment next month
0,1,20000,2,2,1,24,689,0,0,0,576.0,0.034450,0.000000,0.000000,0.000000,0.034450,0.000000,0,0,0,0,0.000000,0.000000,0,-1.0,0.0,0.017225,-0.034450,0.024360,0,1.414214,0.000000,0.000000,0,0,0.000000,0.000000,0.017225,-0.034450,1
1,2,120000,2,2,2,26,2682,3272,1000,1000,676.0,0.022350,0.027267,0.008333,0.008333,0.022350,0.027267,0,0,0,0,0.372856,0.305623,0,0.0,0.0,0.024808,0.004917,0.003477,0,0.140139,0.047541,0.046445,0,0,0.008333,0.000000,0.024808,0.004917,1
2,3,90000,2,2,2,34,13559,14331,1000,1000,1156.0,0.150656,0.159233,0.011111,0.011111,0.150656,0.159233,0,0,0,0,0.073752,0.069779,0,0.0,0.0,0.154944,0.008578,0.006065,0,0.039146,0.002809,0.002805,0,0,0.011111,0.000000,0.154944,0.008578,0
3,4,50000,2,2,1,37,49291,28314,1200,1100,1369.0,0.985820,0.566280,0.024000,0.022000,0.985820,0.566280,0,0,0,0,0.024345,0.038850,0,0.0,0.0,0.77605,-0.419540,0.296660,0,0.382269,0.010256,0.010204,0,0,0.023000,0.001414,0.776050,-0.419540,0
4,5,50000,1,2,1,57,35835,20940,10000,9000,3249.0,0.716700,0.418800,0.200000,0.180000,0.716700,0.418800,0,0,0,0,0.279057,0.429799,0,-0.5,0.707107,0.56775,-0.297900,0.210647,0,0.371021,0.106591,0.101284,0,0,0.190000,0.014142,0.567750,-0.297900,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,220000,1,3,1,39,208365,88004,5003,3047,1521.0,0.947114,0.400018,0.022741,0.013850,0.947114,0.400018,0,0,0,0,0.024011,0.034623,0,0.0,0.0,0.673566,-0.547095,0.386855,0,0.574339,0.007504,0.007476,0,0,0.018295,0.006287,0.673566,-0.547095,0
29996,29997,150000,1,3,2,43,3502,8979,8998,129,1849.0,0.023347,0.059860,0.059987,0.000860,0.023347,0.059860,0,0,1,0,1.000000,0.014367,0,-1.0,0.0,0.041603,0.036513,0.025819,0,0.620595,1.806673,1.032000,0,1,0.030423,0.041809,0.041603,0.036513,0
29997,29998,30000,1,2,2,37,2758,20878,22000,4200,1369.0,0.091933,0.695933,0.733333,0.140000,0.091933,0.695933,0,0,1,0,1.000000,0.201169,1,0.5,2.12132,0.393933,0.604000,0.427092,0,1.084175,5.498198,1.871525,0,1,0.436667,0.419550,0.393933,0.604000,1
29998,29999,80000,1,3,1,41,76304,52774,1178,1926,1681.0,0.953800,0.659675,0.014725,0.024075,0.953800,0.659675,0,0,0,0,0.015438,0.036495,0,0.0,0.0,0.806737,-0.294125,0.207978,0,0.257801,0.014890,0.014780,0,0,0.019400,0.006611,0.806737,-0.294125,1


In [236]:
df_features_b3

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,BILL_AMT5,BILL_AMT6,PAY_AMT5,PAY_AMT6,AGE_SQ,UTILIZACAO_5,UTILIZACAO_6,PROP_PAGAMENTO_5,PROP_PAGAMENTO_6,ENDIVIDAMENTO_5,ENDIVIDAMENTO_6,PROP_NEGATIVA_FLAG_5,PROP_NEGATIVA_FLAG_6,PROP_SUPERIOR_FLAG_5,PROP_SUPERIOR_FLAG_6,PROP_PAGAMENTO_TRAT_5,PROP_PAGAMENTO_TRAT_6,ATRASOS_CONTAGEM_B3,ATRASO_MEDIO_B3,ATRASO_VOLATIL_B3,UTIL_B3,TENDENCIA_UTIL_B3,UTIL_VOLATIL_B3,UTIL_ZERO_B3,CONCENTRACAO_UTIL_B3,PROP_PAGAMENTO_VOLATIL_B3,PROP_PAGAMENTO_VOLATIL_LOG_B3,PROP_NEGATIVA_FLAG_B3,PROP_SUPERIOR_FLAG_B3,MEDIA_PROP_PAGAMENTO_B3,VOL_PROP_PAGAMENTO_B3,MEDIA_ENDIVIDAMENTO_B3,TEND_ENDIVIDAMENTO_B3,default payment next month
0,1,20000,2,2,1,24,0,0,0,0,576.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0,0,0.000000,0.000000,0,-2.0,0.0,0.0,0.000000,0.000000,1,0.000000,0.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,1
1,2,120000,2,2,2,26,3455,3261,0,2000,676.0,0.028792,0.027175,0.000000,0.016667,0.028792,0.027175,0,0,0,0,0.000000,0.613309,1,1.0,1.414214,0.027983,-0.001617,0.001143,0,0.040851,0.433675,0.360241,0,0,0.008333,0.011785,0.027983,-0.001617,1
2,3,90000,2,2,2,34,14948,15549,1000,5000,1156.0,0.166089,0.172767,0.011111,0.055556,0.166089,0.172767,0,0,0,0,0.066899,0.321564,0,0.0,0.0,0.169428,0.006678,0.004722,0,0.027870,0.180076,0.165579,0,0,0.033333,0.031427,0.169428,0.006678,0
3,4,50000,2,2,1,37,28959,29547,1069,1000,1369.0,0.579180,0.590940,0.021380,0.020000,0.579180,0.590940,0,0,0,0,0.036914,0.033844,0,0.0,0.0,0.58506,0.011760,0.008316,0,0.014213,0.002171,0.002168,0,0,0.020690,0.000976,0.585060,0.011760,0
4,5,50000,1,2,1,57,19146,19131,689,679,3249.0,0.382920,0.382620,0.013780,0.013580,0.382920,0.382620,0,0,0,0,0.035987,0.035492,0,0.0,0.0,0.38277,-0.000300,0.000212,0,0.000554,0.000350,0.000350,0,0,0.013680,0.000141,0.382770,-0.000300,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,220000,1,3,1,39,31237,15980,5000,1000,1521.0,0.141986,0.072636,0.022727,0.004545,0.141986,0.072636,0,0,0,0,0.160067,0.062578,0,0.0,0.0,0.107311,-0.069350,0.049038,0,0.456968,0.068935,0.066663,0,0,0.013636,0.012856,0.107311,-0.069350,0
29996,29997,150000,1,3,2,43,5190,0,0,0,1849.0,0.034600,0.000000,0.000000,0.000000,0.034600,0.000000,0,0,0,0,0.000000,0.000000,0,0.0,0.0,0.0173,-0.034600,0.024466,0,1.414214,0.000000,0.000000,0,0,0.000000,0.000000,0.017300,-0.034600,0
29997,29998,30000,1,2,2,37,20582,19357,2000,3100,1369.0,0.686067,0.645233,0.066667,0.103333,0.686067,0.645233,0,0,0,0,0.097172,0.160149,0,0.0,0.0,0.66565,-0.040833,0.028874,0,0.043376,0.044531,0.043568,0,0,0.085000,0.025927,0.665650,-0.040833,1
29998,29999,80000,1,3,1,41,11855,48944,52964,1804,1681.0,0.148187,0.611800,0.662050,0.022550,0.148187,0.611800,0,0,1,0,1.000000,0.036858,0,-0.5,0.707107,0.379994,0.463612,0.327824,0,0.862708,3.133043,1.419014,0,1,0.342300,0.452195,0.379994,0.463612,1


##### INTERPRETAÇÃO DOS BLOCOS
df_features_b1 → meses 1 e 2 (mais recentes)
df_features_b2 → meses 3 e 4 (intermediário)
df_features_b3 → meses 5 e 6 (mais antigos)

B1	M1–M2	comportamento mais recente
B2	M3–M4	tendência intermediária
B3	M5–M6	histórico mais antigo

B3 → treino  
B2 → validação  
B1 → teste (OOT)

treino_antigo → base usada pra treinar
validacao_intermediario → ajuste de modelo
teste_recente → OOT

In [248]:
df_features_b3.to_csv(r"C:\Users\agcsd\Documents\df_features_treino_antigo_M5_M6.csv", index=False)
df_features_b2.to_csv(r"C:\Users\agcsd\Documents\df_features_validacao_intermediario_M3_M4.csv", index=False)
df_features_b1.to_csv(r"C:\Users\agcsd\Documents\df_features_teste_recente_M1_M2.csv", index=False)

In [15]:
df_features_b3 = pd.read_csv(r"C:\Users\agcsd\Documents\df_features_treino_antigo_M5_M6.csv")
df_features_b2 = pd.read_csv(r"C:\Users\agcsd\Documents\df_features_validacao_intermediario_M3_M4.csv")
df_features_b1 = pd.read_csv(r"C:\Users\agcsd\Documents\df_features_teste_recente_M1_M2.csv")

In [17]:
df_features_b3.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,BILL_AMT5,BILL_AMT6,PAY_AMT5,PAY_AMT6,...,CONCENTRACAO_UTIL_B3,PROP_PAGAMENTO_VOLATIL_B3,PROP_PAGAMENTO_VOLATIL_LOG_B3,PROP_NEGATIVA_FLAG_B3,PROP_SUPERIOR_FLAG_B3,MEDIA_PROP_PAGAMENTO_B3,VOL_PROP_PAGAMENTO_B3,MEDIA_ENDIVIDAMENTO_B3,TEND_ENDIVIDAMENTO_B3,default payment next month
0,1,20000,2,2,1,24,0,0,0,0,...,0.000000,0.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,1
1,2,120000,2,2,2,26,3455,3261,0,2000,...,0.040851,0.433675,0.360241,0,0,0.008333,0.011785,0.027983,-0.001617,1
2,3,90000,2,2,2,34,14948,15549,1000,5000,...,0.027870,0.180076,0.165579,0,0,0.033333,0.031427,0.169428,0.006678,0
3,4,50000,2,2,1,37,28959,29547,1069,1000,...,0.014213,0.002171,0.002168,0,0,0.020690,0.000976,0.585060,0.011760,0
4,5,50000,1,2,1,57,19146,19131,689,679,...,0.000554,0.000350,0.000350,0,0,0.013680,0.000141,0.382770,-0.000300,0


In [19]:
df_features_b2.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,BILL_AMT3,BILL_AMT4,PAY_AMT3,PAY_AMT4,...,CONCENTRACAO_UTIL_B2,PROP_PAGAMENTO_VOLATIL_B2,PROP_PAGAMENTO_VOLATIL_LOG_B2,PROP_NEGATIVA_FLAG_B2,PROP_SUPERIOR_FLAG_B2,MEDIA_PROP_PAGAMENTO_B2,VOL_PROP_PAGAMENTO_B2,MEDIA_ENDIVIDAMENTO_B2,TEND_ENDIVIDAMENTO_B2,default payment next month
0,1,20000,2,2,1,24,689,0,0,0,...,1.414214,0.000000,0.000000,0,0,0.000000,0.000000,0.017225,-0.034450,1
1,2,120000,2,2,2,26,2682,3272,1000,1000,...,0.140139,0.047541,0.046445,0,0,0.008333,0.000000,0.024808,0.004917,1
2,3,90000,2,2,2,34,13559,14331,1000,1000,...,0.039146,0.002809,0.002805,0,0,0.011111,0.000000,0.154944,0.008578,0
3,4,50000,2,2,1,37,49291,28314,1200,1100,...,0.382269,0.010256,0.010204,0,0,0.023000,0.001414,0.776050,-0.419540,0
4,5,50000,1,2,1,57,35835,20940,10000,9000,...,0.371021,0.106591,0.101284,0,0,0.190000,0.014142,0.567750,-0.297900,0


In [21]:
df_features_b1.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,BILL_AMT1,BILL_AMT2,PAY_AMT1,PAY_AMT2,...,CONCENTRACAO_UTIL_B1,PROP_PAGAMENTO_VOLATIL_B1,PROP_PAGAMENTO_VOLATIL_LOG_B1,PROP_NEGATIVA_FLAG_B1,PROP_SUPERIOR_FLAG_B1,MEDIA_PROP_PAGAMENTO_B1,VOL_PROP_PAGAMENTO_B1,MEDIA_ENDIVIDAMENTO_B1,TEND_ENDIVIDAMENTO_B1,default payment next month
0,1,20000,2,2,1,24,3913,3102,0,689,...,0.163496,0.157059,0.145881,0,0,0.017225,0.024360,0.175375,-0.040550,1
1,2,120000,2,2,2,26,2682,1725,0,1000,...,0.307103,0.409917,0.343531,0,0,0.004167,0.005893,0.018363,-0.007975,1
2,3,90000,2,2,2,34,29239,14027,1518,1500,...,0.497227,0.038905,0.038167,0,0,0.016767,0.000141,0.240367,-0.169022,0
3,4,50000,2,2,1,37,46990,48233,2000,2019,...,0.018461,0.000497,0.000497,0,0,0.040190,0.000269,0.952230,0.024860,0
4,5,50000,1,2,1,57,8617,5670,2000,36681,...,0.291712,4.410375,1.688318,0,1,0.386810,0.490463,0.142870,-0.058940,0


In [25]:
df_features_b1.columns

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'BILL_AMT1',
       'BILL_AMT2', 'PAY_AMT1', 'PAY_AMT2', 'AGE_SQ', 'UTILIZACAO_1',
       'UTILIZACAO_2', 'PROP_PAGAMENTO_1', 'PROP_PAGAMENTO_2',
       'ENDIVIDAMENTO_1', 'ENDIVIDAMENTO_2', 'PROP_NEGATIVA_FLAG_1',
       'PROP_NEGATIVA_FLAG_2', 'PROP_SUPERIOR_FLAG_1', 'PROP_SUPERIOR_FLAG_2',
       'PROP_PAGAMENTO_TRAT_1', 'PROP_PAGAMENTO_TRAT_2', 'ATRASOS_CONTAGEM_B1',
       'ATRASO_MEDIO_B1', 'ATRASO_VOLATIL_B1', 'UTIL_B1', 'TENDENCIA_UTIL_B1',
       'UTIL_VOLATIL_B1', 'UTIL_ZERO_B1', 'CONCENTRACAO_UTIL_B1',
       'PROP_PAGAMENTO_VOLATIL_B1', 'PROP_PAGAMENTO_VOLATIL_LOG_B1',
       'PROP_NEGATIVA_FLAG_B1', 'PROP_SUPERIOR_FLAG_B1',
       'MEDIA_PROP_PAGAMENTO_B1', 'VOL_PROP_PAGAMENTO_B1',
       'MEDIA_ENDIVIDAMENTO_B1', 'TEND_ENDIVIDAMENTO_B1',
       'default payment next month'],
      dtype='object')

In [27]:
df_features_b2.columns

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'BILL_AMT3',
       'BILL_AMT4', 'PAY_AMT3', 'PAY_AMT4', 'AGE_SQ', 'UTILIZACAO_3',
       'UTILIZACAO_4', 'PROP_PAGAMENTO_3', 'PROP_PAGAMENTO_4',
       'ENDIVIDAMENTO_3', 'ENDIVIDAMENTO_4', 'PROP_NEGATIVA_FLAG_3',
       'PROP_NEGATIVA_FLAG_4', 'PROP_SUPERIOR_FLAG_3', 'PROP_SUPERIOR_FLAG_4',
       'PROP_PAGAMENTO_TRAT_3', 'PROP_PAGAMENTO_TRAT_4', 'ATRASOS_CONTAGEM_B2',
       'ATRASO_MEDIO_B2', 'ATRASO_VOLATIL_B2', 'UTIL_B2', 'TENDENCIA_UTIL_B2',
       'UTIL_VOLATIL_B2', 'UTIL_ZERO_B2', 'CONCENTRACAO_UTIL_B2',
       'PROP_PAGAMENTO_VOLATIL_B2', 'PROP_PAGAMENTO_VOLATIL_LOG_B2',
       'PROP_NEGATIVA_FLAG_B2', 'PROP_SUPERIOR_FLAG_B2',
       'MEDIA_PROP_PAGAMENTO_B2', 'VOL_PROP_PAGAMENTO_B2',
       'MEDIA_ENDIVIDAMENTO_B2', 'TEND_ENDIVIDAMENTO_B2',
       'default payment next month'],
      dtype='object')

In [23]:
df_features_b3.columns

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'BILL_AMT5',
       'BILL_AMT6', 'PAY_AMT5', 'PAY_AMT6', 'AGE_SQ', 'UTILIZACAO_5',
       'UTILIZACAO_6', 'PROP_PAGAMENTO_5', 'PROP_PAGAMENTO_6',
       'ENDIVIDAMENTO_5', 'ENDIVIDAMENTO_6', 'PROP_NEGATIVA_FLAG_5',
       'PROP_NEGATIVA_FLAG_6', 'PROP_SUPERIOR_FLAG_5', 'PROP_SUPERIOR_FLAG_6',
       'PROP_PAGAMENTO_TRAT_5', 'PROP_PAGAMENTO_TRAT_6', 'ATRASOS_CONTAGEM_B3',
       'ATRASO_MEDIO_B3', 'ATRASO_VOLATIL_B3', 'UTIL_B3', 'TENDENCIA_UTIL_B3',
       'UTIL_VOLATIL_B3', 'UTIL_ZERO_B3', 'CONCENTRACAO_UTIL_B3',
       'PROP_PAGAMENTO_VOLATIL_B3', 'PROP_PAGAMENTO_VOLATIL_LOG_B3',
       'PROP_NEGATIVA_FLAG_B3', 'PROP_SUPERIOR_FLAG_B3',
       'MEDIA_PROP_PAGAMENTO_B3', 'VOL_PROP_PAGAMENTO_B3',
       'MEDIA_ENDIVIDAMENTO_B3', 'TEND_ENDIVIDAMENTO_B3',
       'default payment next month'],
      dtype='object')

In [29]:
df_base.columns

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0',
       'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
       'default payment next month'],
      dtype='object', name=0)